In [ ]:
# ============================================================================================
# PAYER DASHBOARD - FUNCTIONAL TEST QUERIES (R2)    
# ============================================================================================

from datetime import date
from pyspark.sql import functions as F

In [ ]:
SOURCE = "INTERMEDIATE"

ADB_SCHEMA       = "ext_dev_adb.prov_analytics"
SEMANTIC_SCHEMA  = "dev_plexus.semantic_layer"
REFERENCE_SCHEMA = "dev_plexus.ref_shared"
GOLD_SCHEMA      = "dev_plexus.claims_gold"

ORG = 1

FROM_DATE = "2025-12-01"
TO_DATE   = "2026-11-30"

SCOPES_TO_TEST = ["NET", "GROSS"]

PAYER_TYPES = []
PAYER_IDS   = []
CLAIM_TYPES = []

DRILL_PAYER_ID      = 10
FILTER_PAYER_TYPE   = "Commercial"
FILTER_PAYER_ID     = 10
FILTER_CLAIM_TYPE   = "Professional"

CLAIM_IDS = {
    "PD-01":  "",
    "PD-02":  "",
    "PD-03":  "",
    "PD-04":  "",
    "PD-05A": "",
    "PD-05B": "",
    "PD-08":  "",
    "PD-38":  "",
    "PD-04B": "",
    "PD-40":  "",
    "PD-43":  "",
}

EXPECTED_PRECEDENCE = "EMAIL"

DEADLINE_CARC_REGEX = "^(CO|PI)-(226|227|252)$"

In [ ]:
ADB_KPI  = f"{ADB_SCHEMA}.r2_kpi_dashboard_details"
ADB_CAT  = f"{ADB_SCHEMA}.r2_denial_claim_category"
ADB_DPAY = f"{ADB_SCHEMA}.r2_dim_payer"
PCARC    = f"{ADB_SCHEMA}.r2_denial_claim_payer_carc"
KPI_STG  = f"{ADB_SCHEMA}.r2_kpi_dashboard_details_stg"
CAT_STG  = f"{ADB_SCHEMA}.r2_denial_claim_category_stg"

S_KPI  = f"{SEMANTIC_SCHEMA}.denial_kpi_details"
S_CARC = f"{SEMANTIC_SCHEMA}.denial_carc_details"

if SOURCE not in ("INTERMEDIATE", "ADB"):
    raise ValueError("SOURCE must be INTERMEDIATE or ADB")

if SOURCE == "INTERMEDIATE":
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW t_kpi_dashboard_details AS
    WITH n AS (SELECT * FROM {S_KPI} WHERE scope = 'NET'   AND organization_id = {int(ORG)}),
         g AS (SELECT * FROM {S_KPI} WHERE scope = 'GROSS' AND organization_id = {int(ORG)})
    SELECT COALESCE(n.organization_id,        g.organization_id)        AS organization_id,
           COALESCE(n.claim_id,               g.claim_id)               AS claim_id,
           COALESCE(n.claim_version_id,       g.claim_version_id)       AS claim_version_id,
           COALESCE(n.patient_control_number, g.patient_control_number) AS patient_control_number,
           COALESCE(n.production_date,        g.production_date)        AS production_date,
           COALESCE(n.payer_id,               g.payer_id)               AS payer_id,
           COALESCE(n.payer_type,             g.payer_type)             AS payer_type,
           COALESCE(n.claim_type,             g.claim_type)             AS claim_type,
           COALESCE(n.denial_type,            g.denial_type)            AS denial_type,
           COALESCE(n.claim_rollup_type,      g.claim_rollup_type)      AS claim_rollup_type,
           n.claim_charge_amount        AS net_claim_charge_amount,
           g.claim_charge_amount        AS gross_claim_charge_amount,
           CASE WHEN n.actionable_denied_amount IS NOT NULL
                THEN n.claim_charge_amount END AS actionable_claim_charge_amount,
           n.total_denied_amount        AS net_total_denied_amount,
           g.total_denied_amount        AS gross_total_denied_amount,
           n.actionable_denied_amount   AS actionable_total_denied_amount
    FROM n
    FULL OUTER JOIN g
      ON g.organization_id <=> n.organization_id
     AND g.claim_version_id = n.claim_version_id
    """)
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW t_denial_claim_category AS
    WITH cat AS (
      SELECT scope, organization_id, claim_version_id, denial_category,
             SUM(denied_amount) AS denied_amount, SUM(actionable_denied_amount) AS actionable_denied_amount
      FROM {S_CARC}
      WHERE organization_id = {int(ORG)}
      GROUP BY scope, organization_id, claim_version_id, denial_category),
    n AS (SELECT * FROM cat WHERE scope = 'NET'),
    g AS (SELECT * FROM cat WHERE scope = 'GROSS'),
    merged AS (
      SELECT COALESCE(n.organization_id,  g.organization_id)  AS organization_id,
             COALESCE(n.claim_version_id, g.claim_version_id) AS claim_version_id,
             COALESCE(n.denial_category,  g.denial_category)  AS denial_category,
             n.denied_amount AS net_total_denied_amount,
             g.denied_amount AS gross_total_denied_amount,
             n.actionable_denied_amount AS actionable_total_denied_amount
      FROM n FULL OUTER JOIN g
        ON g.organization_id <=> n.organization_id AND g.claim_version_id = n.claim_version_id
       AND g.denial_category <=> n.denial_category),
    kn AS (SELECT * FROM {S_KPI} WHERE scope = 'NET'   AND organization_id = {int(ORG)}),
    kg AS (SELECT * FROM {S_KPI} WHERE scope = 'GROSS' AND organization_id = {int(ORG)})
    SELECT m.organization_id,
           COALESCE(kn.claim_id, kg.claim_id)                             AS claim_id,
           m.claim_version_id,
           COALESCE(kn.patient_control_number, kg.patient_control_number) AS patient_control_number,
           COALESCE(kn.production_date, kg.production_date)               AS production_date,
           COALESCE(kn.payer_id, kg.payer_id)                             AS payer_id,
           COALESCE(kn.payer_type, kg.payer_type)                         AS payer_type,
           COALESCE(kn.claim_type, kg.claim_type)                         AS claim_type,
           m.denial_category, m.net_total_denied_amount, m.gross_total_denied_amount, m.actionable_total_denied_amount
    FROM merged m
    LEFT JOIN kn ON kn.organization_id <=> m.organization_id AND kn.claim_version_id = m.claim_version_id
    LEFT JOIN kg ON kg.organization_id <=> m.organization_id AND kg.claim_version_id = m.claim_version_id
    """)
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW t_dim_payer AS
    SELECT CAST(payer_sk AS BIGINT) AS payer_id, MAX(payer_name) AS payer_name
    FROM {SEMANTIC_SCHEMA}.dim_payer
    WHERE record_active_indicator = 1
    GROUP BY CAST(payer_sk AS BIGINT)
    """)
    KPI, CAT, DPAY = "t_kpi_dashboard_details", "t_denial_claim_category", "t_dim_payer"
else:
    KPI, CAT, DPAY = ADB_KPI, ADB_CAT, ADB_DPAY

print(f"widget queries read: {KPI} | {CAT} | {DPAY}")

SCOPE_COLUMN = {
    "NET": "net_total_denied_amount",
    "ACTIONABLE": "actionable_total_denied_amount",
    "GROSS": "gross_total_denied_amount",
}
SCOPE_LABEL = {"NET": "All denials", "ACTIONABLE": "Actionable denials only", "GROSS": "Cumulative submissions"}
if not SCOPES_TO_TEST or any(s not in ("NET", "GROSS") for s in SCOPES_TO_TEST):
    raise ValueError("SCOPES_TO_TEST must list NET and / or GROSS")
SCOPE = SCOPES_TO_TEST[0]


def amt(scope=None):
    """Denied-amount column of a scope in the ADB targets."""
    return SCOPE_COLUMN[scope or SCOPE]


def denied_cond(scope=None, alias="k"):
    """A claim version is a denied claim for the scope."""
    return f"{alias}.claim_rollup_type = 'Denial' AND {alias}.{amt(scope)} IS NOT NULL"


def sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" if isinstance(v, str) else str(v) for v in values)


def where(alias="k", date_from=None, date_to=None, payer_types=None, payer_ids=None, claim_types=None):
    """Dashboard filters: organization, production_date range, payer type, payer, claim type."""
    payer_types = PAYER_TYPES if payer_types is None else payer_types
    payer_ids = PAYER_IDS if payer_ids is None else payer_ids
    claim_types = CLAIM_TYPES if claim_types is None else claim_types
    parts = [
        f"{alias}.organization_id = {int(ORG)}",
        f"{alias}.production_date BETWEEN DATE'{date_from or FROM_DATE}' AND DATE'{date_to or TO_DATE}'",
    ]
    if payer_types:
        parts.append(f"{alias}.payer_type IN ({sql_list(payer_types)})")
    if payer_ids:
        parts.append(f"{alias}.payer_id IN ({sql_list(payer_ids)})")
    if claim_types:
        parts.append(f"{alias}.claim_type IN ({sql_list(claim_types)})")
    return "WHERE " + "\n  AND ".join(parts)


RESULTS = []


def record(test_id, purpose, status, rows=None, detail=""):
    RESULTS.append((test_id, purpose, status, rows, detail[:500]))


def claim(key):
    """Claim id for a scenario case, or None when neither a preset nor a candidate exists."""
    value = (CLAIM_IDS.get(key) or "").strip()
    return value.replace("'", "''") or None


def pick(key, df, column="claim_id"):
    """Use the first candidate of a find cell unless a claim is preset in CLAIM_IDS."""
    if (CLAIM_IDS.get(key) or "").strip():
        print(f"{key}: using preset claim {CLAIM_IDS[key]}")
        return
    if df is None:
        print(f"{key}: no candidate (the find query did not run)")
        return
    rows = df.select(column).limit(1).collect()
    if rows:
        CLAIM_IDS[key] = rows[0][0]
        print(f"{key}: using first candidate {CLAIM_IDS[key]}")
    else:
        print(f"{key}: no candidate claim in this data")


def show(test_id, purpose, sql, expect=None, expect_zero_rows=False, limit=200):
    """Print the query, run it and display the result. Never raises: a failure is reported
    as ERROR so that a full run continues to the next cell."""
    print(f"{test_id} | {purpose}")
    if expect:
        print(f"Expected: {expect}")
    print("-" * 100)
    print(sql.strip())
    print("-" * 100)
    try:
        df = spark.sql(sql)
        rows = df.count()
    except Exception as exc:
        message = str(exc).strip().split("\n")[0]
        print(f"RESULT: ERROR - {message}")
        record(test_id, purpose, "ERROR", None, message)
        return None
    print(f"rows returned: {rows}")
    status = "OK"
    if expect_zero_rows:
        status = "OK" if rows == 0 else "CHECK"
        print("RESULT: as expected (0 rows)" if rows == 0 else "RESULT: CHECK - rows returned where 0 were expected")
    record(test_id, purpose, status, rows)
    try:
        display(df.limit(limit))
    except NameError:
        df.show(limit, truncate=False)
    return df


def skipped(test_id, purpose, reason):
    print(f"{test_id} | {purpose}\nRESULT: SKIPPED - {reason}")
    record(test_id, purpose, "SKIPPED", None, reason)


def visual(test_id, purpose):
    print(f"{test_id} | VISUAL CHECK - {purpose}")
    record(test_id, purpose, "VISUAL", None, "Check on the dashboard; no query.")


G_SUM  = f"{GOLD_SCHEMA}.claim_payment_summary_gross"
G_CADJ = f"{GOLD_SCHEMA}.claim_payment_summary_adjustment_gross"
G_LINE = f"{GOLD_SCHEMA}.claim_payment_service_line_gross"
G_LADJ = f"{GOLD_SCHEMA}.claim_payment_service_line_adjustment_gross"


def _columns(table):
    try:
        return [c.lower() for c in spark.table(table).columns]
    except Exception:
        return None


def _col(cols, alias, *names, default="NULL"):
    """First existing column among names, qualified with alias; default when none exists."""
    for n in names:
        if cols and n in cols:
            return f"{alias}.{n}"
    return default


def _empty_gold_views(reason):
    spark.sql("""CREATE OR REPLACE TEMP VIEW g_versions AS SELECT
        CAST(NULL AS STRING) claim_id, CAST(NULL AS STRING) claim_version_id, CAST(NULL AS DATE) production_dt,
        CAST(NULL AS STRING) claim_status_code, CAST(NULL AS DECIMAL(18,2)) claim_charge,
        CAST(NULL AS DECIMAL(18,2)) payment_amount, CAST(NULL AS STRING) payer_name_norm,
        CAST(NULL AS STRING) claim_filing_cd, CAST(NULL AS STRING) pccn_norm, CAST(NULL AS STRING) version_ts,
        CAST(NULL AS STRING) rec_insert_ts, CAST(NULL AS STRING) header_id WHERE 1 = 0""")
    spark.sql("""CREATE OR REPLACE TEMP VIEW g_cas AS SELECT
        CAST(NULL AS STRING) claim_version_id, CAST(NULL AS STRING) cas_level, CAST(NULL AS STRING) line_id,
        CAST(NULL AS DECIMAL(18,2)) line_charge, CAST(NULL AS STRING) carc_key, CAST(NULL AS DECIMAL(18,2)) amt,
        CAST(NULL AS INT) in_seed, CAST(NULL AS INT) seed_denial WHERE 1 = 0""")
    print(f"GOLD GROSS NOT AVAILABLE ({reason}) - Gold-based checks will return no rows")


GOLD_OK = False
try:
    cs, cc, cl, ca = _columns(G_SUM), _columns(G_CADJ), _columns(G_LINE), _columns(G_LADJ)
    if not (cs and cc and cl and ca):
        raise ValueError("one of the Gold Gross tables is missing")

    try:
        spark.sql(f"SELECT client_id, client_name FROM {REFERENCE_SCHEMA}.dim_client LIMIT 1").collect()
        org_filter = (f"UPPER(TRIM(s.provider_name)) IN (SELECT UPPER(TRIM(client_name)) FROM {REFERENCE_SCHEMA}.dim_client "
                      f"WHERE client_id = {int(ORG)} AND record_active_indicator = 1)")
        ORG_FILTER_NOTE = "provider_name -> dim_client"
    except Exception:
        org_filter = f"s.claim_payment_id IN (SELECT claim_id FROM {S_KPI} WHERE organization_id = {int(ORG)})"
        ORG_FILTER_NOTE = "claims present in denial_kpi_details (dim_client not readable)"

    prod = _col(cs, "s", "production_date")
    pay_dt = _col(cs, "s", "payment_date")
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW g_versions AS
    SELECT s.claim_payment_id                                         AS claim_id,
           s.claim_payment_claimlevel_id                              AS claim_version_id,
           COALESCE(TO_DATE(CAST({prod} AS STRING), 'yyyyMMdd'), TO_DATE(CAST({prod} AS STRING)),
                    TO_DATE(CAST({pay_dt} AS STRING), 'yyyyMMdd'), TO_DATE(CAST({pay_dt} AS STRING))) AS production_dt,
           CAST({_col(cs, 's', 'claim_status_code')} AS STRING)       AS claim_status_code,
           CAST({_col(cs, 's', 'total_claim_charge_amount')} AS DECIMAL(18,2)) AS claim_charge,
           CAST({_col(cs, 's', 'total_payment_amount', 'claim_payment_amount')} AS DECIMAL(18,2)) AS payment_amount,
           UPPER(TRIM({_col(cs, 's', 'payer_name')}))                 AS payer_name_norm,
           UPPER(TRIM({_col(cs, 's', 'claim_filing_indicator_code')})) AS claim_filing_cd,
           NULLIF(UPPER(TRIM({_col(cs, 's', 'payer_claim_control_number')})), '') AS pccn_norm,
           CAST({_col(cs, 's', 'version_timestamp')} AS STRING)       AS version_ts,
           CAST({_col(cs, 's', 'rec_insert_datetime')} AS STRING)     AS rec_insert_ts,
           CAST({_col(cs, 's', 'claim_payment_header_id')} AS STRING) AS header_id
    FROM {G_SUM} s
    WHERE {org_filter}
    """)

    l_grp = _col(ca, "a", "line_adjustment_group_code", "line_adjustment_group")
    l_rsn = _col(ca, "a", "line_adjustment_reason_code")
    l_amt = _col(ca, "a", "line_adjustment_amount")
    c_grp = _col(cc, "a", "adjustment_group_code", "adjustment_group")
    c_rsn = _col(cc, "a", "adjustment_reason_code")
    c_amt = _col(cc, "a", "adjustment_amount")
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW g_cas AS
    WITH seed AS (
      SELECT UPPER(TRIM(carc_code)) AS carc_key, MAX(UPPER(TRIM(denial_ind))) AS denial_ind
      FROM {REFERENCE_SCHEMA}.dim_carc WHERE client_id = {int(ORG)} AND record_active_indicator = 1
      GROUP BY UPPER(TRIM(carc_code))),
    raw AS (
      SELECT a.claim_payment_claimlevel_id AS claim_version_id, 'LINE' AS cas_level,
             CAST(a.claim_payment_service_line_id AS STRING) AS line_id,
             CAST(l.line_charge_amount AS DECIMAL(18,2)) AS line_charge,
             CONCAT(UPPER(TRIM({l_grp})), '-', UPPER(TRIM({l_rsn}))) AS carc_key,
             CAST({l_amt} AS DECIMAL(18,2)) AS amt
      FROM {G_LADJ} a
      LEFT JOIN {G_LINE} l ON l.claim_payment_service_line_id = a.claim_payment_service_line_id
      WHERE a.claim_payment_claimlevel_id IN (SELECT claim_version_id FROM g_versions)
      UNION ALL
      SELECT a.claim_payment_claimlevel_id, 'CLAIM', CAST(NULL AS STRING), CAST(NULL AS DECIMAL(18,2)),
             CONCAT(UPPER(TRIM({c_grp})), '-', UPPER(TRIM({c_rsn}))), CAST({c_amt} AS DECIMAL(18,2))
      FROM {G_CADJ} a
      WHERE a.claim_payment_claimlevel_id IN (SELECT claim_version_id FROM g_versions))
    SELECT r.*, CASE WHEN s.carc_key IS NULL THEN 0 ELSE 1 END AS in_seed,
           CASE WHEN s.denial_ind = 'Y' THEN 1 ELSE 0 END AS seed_denial
    FROM raw r LEFT JOIN seed s ON s.carc_key = r.carc_key
    """)
    GOLD_OK = True
except Exception as exc:
    _empty_gold_views(str(exc).strip().split("\n")[0][:200])
    ORG_FILTER_NOTE = "n/a"

spark.sql("""
CREATE OR REPLACE TEMP VIEW g_expected AS
WITH lvl AS (
  SELECT claim_version_id, CASE WHEN MAX(CASE WHEN cas_level = 'LINE' THEN 1 ELSE 0 END) = 1 THEN 'LINE' ELSE 'CLAIM' END AS selected_level
  FROM g_cas GROUP BY claim_version_id),
sel AS (SELECT c.* FROM g_cas c JOIN lvl ON lvl.claim_version_id = c.claim_version_id AND lvl.selected_level = c.cas_level),
line_tot AS (
  SELECT claim_version_id, line_id, MAX(line_charge) AS line_charge, SUM(CASE WHEN seed_denial = 1 THEN amt ELSE 0 END) AS line_denied,
         MAX(seed_denial) AS has_denial
  FROM sel WHERE cas_level = 'LINE' GROUP BY claim_version_id, line_id),
line_breach AS (
  SELECT claim_version_id,
         MAX(CASE WHEN has_denial = 1 AND (line_charge IS NULL OR line_charge = 0 OR line_denied > line_charge) THEN 1 ELSE 0 END) AS line_rule_breach
  FROM line_tot GROUP BY claim_version_id)
SELECT v.claim_id, v.claim_version_id, v.production_dt, v.claim_status_code, v.claim_charge, v.payment_amount,
       COALESCE(lvl.selected_level, 'NONE')                                                         AS selected_level,
       CAST(COALESCE(SUM(CASE WHEN sel.seed_denial = 1 THEN sel.amt END), 0) AS DECIMAL(18,2))      AS seed_denied_amount,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CASE WHEN sel.seed_denial = 1 THEN CONCAT(sel.carc_key, ' ', CAST(sel.amt AS STRING)) END))) AS seed_denial_cas,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CONCAT(sel.carc_key, ' ', CAST(sel.amt AS STRING)))))   AS selected_cas,
       MAX(CASE WHEN sel.amt < 0 THEN 1 ELSE 0 END)                                                AS has_negative_cas,
       MAX(CASE WHEN sel.seed_denial = 1 AND sel.amt < 0 THEN 1 ELSE 0 END)                        AS has_negative_denial_cas,
       MAX(CASE WHEN sel.in_seed = 0 THEN 1 ELSE 0 END)                                            AS has_cas_not_in_seed,
       MAX(sel.in_seed)                                                                           AS any_in_seed,
       COUNT(sel.carc_key)                                                                        AS selected_cas_count,
       COALESCE(MAX(lb.line_rule_breach), 0)                                                       AS line_rule_breach
FROM g_versions v
LEFT JOIN lvl ON lvl.claim_version_id = v.claim_version_id
LEFT JOIN sel ON sel.claim_version_id = v.claim_version_id
LEFT JOIN line_breach lb ON lb.claim_version_id = v.claim_version_id
GROUP BY v.claim_id, v.claim_version_id, v.production_dt, v.claim_status_code, v.claim_charge, v.payment_amount, lvl.selected_level
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW n_sel AS
SELECT claim_id, claim_version_id FROM {S_KPI} WHERE organization_id = {int(ORG)} AND scope = 'NET'
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW g_net_candidates AS
SELECT e.* FROM g_expected e JOIN n_sel n ON n.claim_version_id = e.claim_version_id
WHERE COALESCE(e.payment_amount, 0) >= 0 AND COALESCE(e.claim_charge, 0) > 0
""")

print(f"Gold Gross available: {GOLD_OK} | organization filter on Gold: {ORG_FILTER_NOTE}")


def trace_claim(test_id, key, expect):
    """KPI and category rows of a claim next to the Gold Gross expectation of each version."""
    cid = claim(key)
    if cid is None:
        skipped(test_id, f"trace claim ({key})", "no claim preset and the find cell returned no candidate")
        return None
    return show(test_id, f"claim {cid} - KPI / category rows ({SOURCE}) and Gold Gross versions", f"""
SELECT 'KPI' AS target, k.claim_version_id,
       CASE WHEN n.claim_version_id IS NOT NULL THEN 'Y' ELSE 'N' END AS is_net_version,
       CAST(k.production_date AS DATE) AS production_date, CAST(k.payer_id AS BIGINT) AS payer_id, k.claim_type,
       k.denial_type, k.claim_rollup_type, CAST(NULL AS STRING) AS denial_category,
       k.net_total_denied_amount, k.actionable_total_denied_amount, k.gross_total_denied_amount,
       e.selected_level, e.seed_denied_amount AS expected_seed_denied, e.seed_denial_cas, e.claim_charge, e.payment_amount, e.claim_status_code
FROM {KPI} k
LEFT JOIN n_sel n ON n.claim_version_id = k.claim_version_id
LEFT JOIN g_expected e ON e.claim_version_id = k.claim_version_id
WHERE k.organization_id = {ORG} AND k.claim_id = '{cid}'
UNION ALL
SELECT 'CATEGORY', c.claim_version_id, CAST(NULL AS STRING), CAST(c.production_date AS DATE), CAST(c.payer_id AS BIGINT), c.claim_type,
       CAST(NULL AS STRING), CAST(NULL AS STRING), c.denial_category,
       c.net_total_denied_amount, c.actionable_total_denied_amount, c.gross_total_denied_amount,
       CAST(NULL AS STRING), CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS STRING), CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS STRING)
FROM {CAT} c WHERE c.organization_id = {ORG} AND c.claim_id = '{cid}'
UNION ALL
SELECT 'GOLD_VERSION_NOT_IN_KPI', e.claim_version_id, 'N', e.production_dt, CAST(NULL AS BIGINT), CAST(NULL AS STRING),
       CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS DECIMAL(18,2)), CAST(NULL AS DECIMAL(18,2)),
       e.selected_level, e.seed_denied_amount, e.seed_denial_cas, e.claim_charge, e.payment_amount, e.claim_status_code
FROM g_expected e
WHERE e.claim_id = '{cid}'
  AND e.claim_version_id NOT IN (SELECT claim_version_id FROM {KPI} WHERE organization_id = {ORG} AND claim_id = '{cid}')
ORDER BY 1, 2
""", expect=expect)


TODAY = date.today()
MTD_FROM = TODAY.replace(day=1).isoformat()
T12_FROM = date(TODAY.year - (1 if TODAY.month < 12 else 0), (TODAY.month % 12) + 1, 1).isoformat()
TO_TODAY = TODAY.isoformat()

print(f"organization {ORG} | {FROM_DATE} to {TO_DATE} | scopes tested: "
      + ", ".join(f"{s} ({SCOPE_LABEL[s]}) -> {amt(s)}" for s in SCOPES_TO_TEST))
print(f"MTD from {MTD_FROM} | T12 from {T12_FROM} | today {TO_TODAY}")

In [ ]:
_candidates = show("PD-01", "selected Net versions with CO-45, a PR CAS and a seed-denial CAS at service-line level", f"""
SELECT e.claim_id, e.claim_version_id, e.selected_cas, e.seed_denial_cas, e.seed_denied_amount, e.claim_charge
FROM g_net_candidates e
WHERE e.selected_level = 'LINE' AND e.has_negative_cas = 0 AND e.has_cas_not_in_seed = 0 AND e.line_rule_breach = 0
  AND e.seed_denied_amount > 0 AND e.seed_denied_amount <= e.claim_charge
  AND EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = 'LINE' AND c.carc_key = 'CO-45')
  AND EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = 'LINE' AND c.carc_key LIKE 'PR-%')
ORDER BY e.selected_cas_count, e.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-01'] is set.", limit=50)
pick("PD-01", _candidates)

In [ ]:
trace_claim("PD-01", "PD-01", "Net version: net_total_denied_amount = expected_seed_denied (seed-denial CAS only; CO-45 and PR amounts not counted). Category rows only for the denial CARCs.")

In [ ]:
_candidates = show("PD-02", "selected Net versions with a positive service-line denial and a positive claim-level denial", f"""
SELECT e.claim_id, e.claim_version_id, e.seed_denial_cas AS line_denial_cas, e.seed_denied_amount AS line_seed_denied,
       (SELECT CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(CONCAT(c.carc_key, ' ', CAST(c.amt AS STRING)))))
          FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = 'CLAIM') AS claim_level_cas,
       e.claim_charge
FROM g_net_candidates e
WHERE e.selected_level = 'LINE' AND e.has_negative_cas = 0 AND e.line_rule_breach = 0 AND e.seed_denied_amount > 0
  AND e.seed_denied_amount <= e.claim_charge
  AND EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = 'CLAIM' AND c.seed_denial = 1 AND c.amt > 0)
  AND NOT EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.amt < 0)
ORDER BY e.selected_cas_count, e.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-02'] is set.", limit=50)
pick("PD-02", _candidates)

In [ ]:
trace_claim("PD-02", "PD-02", "net_total_denied_amount = expected_seed_denied (service-line denial CAS only); the claim-level denial amount is not added.")

In [ ]:
_candidates = show("PD-03", "selected Net versions whose selected CAS are all missing from the seed", f"""
SELECT e.claim_id, e.claim_version_id, e.selected_level, e.selected_cas, e.claim_charge
FROM g_net_candidates e
WHERE e.selected_cas_count > 0 AND e.any_in_seed = 0
ORDER BY e.selected_cas_count, e.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-03'] is set.", limit=50)
pick("PD-03", _candidates)

In [ ]:
trace_claim("PD-03", "PD-03", "Net version: claim_rollup_type Non-Denial, denial_type Accepted, net 0.00, no CATEGORY row (unless a CARC-RARC refinement qualifies the CAS).")

In [ ]:
CID = claim("PD-03")
REVIEW_TABLE = f"{SEMANTIC_SCHEMA}.denial_claims_review"
_rcols = _columns(REVIEW_TABLE)
if CID is None:
    skipped("PD-03", "review flags", "no claim preset and the find cell returned no candidate")
elif not _rcols:
    skipped("PD-03", "review flags", f"{REVIEW_TABLE} is not available")
else:
    _org = f"organization_id = {ORG} AND " if "organization_id" in _rcols else ""
    show("PD-03", "denial_claims_review rows for claim " + CID, f"""
    SELECT * FROM {REVIEW_TABLE}
    WHERE {_org}claim_id = '{CID}'
    """, expect="A CARC_NOT_IN_SEED review row with outcome EXCLUDED for each CAS of the claim (column names follow the table's current layout).")

In [ ]:
show("PD-04", "surviving Gold Gross versions by reversal-related status", f"""
SELECT claim_status_code, COUNT(*) AS versions,
       SUM(CASE WHEN payment_amount < 0 THEN 1 ELSE 0 END) AS negative_payment_versions,
       SUM(CASE WHEN claim_version_id IN (SELECT claim_version_id FROM {S_KPI} WHERE organization_id = {ORG}) THEN 1 ELSE 0 END) AS versions_in_kpi
FROM g_versions
WHERE claim_status_code IN ('22', '23')
GROUP BY claim_status_code
ORDER BY claim_status_code
""", expect="Status 22: 0 versions (reversals removed by Gross Gold). Status 23 rows, if any, are reported for the open point on status 23.")

In [ ]:
_candidates = show("PD-04", "claims with a negative-payment Gross version", f"""
SELECT e.claim_id, e.claim_version_id AS negative_payment_version, e.claim_status_code, e.payment_amount, e.seed_denial_cas,
       (SELECT COUNT(*) FROM g_versions v WHERE v.claim_id = e.claim_id) AS gross_versions
FROM g_expected e
WHERE e.payment_amount < 0 AND COALESCE(e.claim_status_code, '') <> '22'
ORDER BY CASE WHEN e.seed_denied_amount <> 0 THEN 0 ELSE 1 END, e.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-04B'] is set.", limit=50)
pick("PD-04B", _candidates)

In [ ]:
trace_claim("PD-04", "PD-04B", "The negative-payment version has no KPI or CATEGORY row in any scope (GOLD_VERSION_NOT_IN_KPI). If the claim has other versions, the Net version is one of them.")

In [ ]:
_candidates = show("PD-05", "selected Net versions whose only seed-denial CARC is CO-197", f"""
SELECT e.claim_id, e.claim_version_id, e.seed_denial_cas, e.seed_denied_amount, e.claim_charge
FROM g_net_candidates e
WHERE e.has_negative_cas = 0 AND e.line_rule_breach = 0 AND e.seed_denied_amount > 0 AND e.seed_denied_amount <= e.claim_charge
  AND EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = e.selected_level AND c.seed_denial = 1 AND c.carc_key = 'CO-197')
  AND NOT EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = e.selected_level AND c.seed_denial = 1 AND c.carc_key <> 'CO-197')
ORDER BY e.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-05A'] is set.", limit=50)
pick("PD-05A", _candidates)

In [ ]:
_candidates = show("PD-05", "selected Net versions whose seed denials are all deadline CARCs past the payer timely-filing period", f"""
WITH d AS (
  SELECT e.claim_id, e.claim_version_id, e.production_dt, e.seed_denial_cas, e.seed_denied_amount, v.payer_name_norm, v.claim_filing_cd
  FROM g_net_candidates e JOIN g_versions v ON v.claim_version_id = e.claim_version_id
  WHERE e.has_negative_cas = 0 AND e.line_rule_breach = 0 AND e.seed_denied_amount > 0 AND e.seed_denied_amount <= e.claim_charge
    AND EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = e.selected_level
                AND c.seed_denial = 1 AND c.carc_key RLIKE '{DEADLINE_CARC_REGEX}')
    AND NOT EXISTS (SELECT 1 FROM g_cas c WHERE c.claim_version_id = e.claim_version_id AND c.cas_level = e.selected_level
                    AND c.seed_denial = 1 AND NOT (c.carc_key RLIKE '{DEADLINE_CARC_REGEX}')))
SELECT d.claim_id, d.claim_version_id, d.seed_denial_cas, d.payer_name_norm, d.production_dt,
       DATEDIFF(CURRENT_DATE(), d.production_dt) AS age_days, MIN(CAST(t.timely_days AS INT)) AS timely_days
FROM d
JOIN {REFERENCE_SCHEMA}.dim_timely_filing t
  ON t.client_id = {ORG} AND t.record_active_indicator = 1
 AND UPPER(TRIM(t.payer_name)) = d.payer_name_norm AND UPPER(TRIM(t.claim_filing_cd)) = d.claim_filing_cd
GROUP BY d.claim_id, d.claim_version_id, d.seed_denial_cas, d.payer_name_norm, d.production_dt
HAVING DATEDIFF(CURRENT_DATE(), d.production_dt) > MIN(CAST(t.timely_days AS INT))
ORDER BY d.claim_id
""", expect="Candidate claims (selected Net version). The first one is traced unless CLAIM_IDS['PD-05B'] is set.", limit=50)
pick("PD-05B", _candidates)

In [ ]:
trace_claim("PD-05", "PD-05A", "Claim A: Net version net = actionable = expected_seed_denied (both populated).")

In [ ]:
trace_claim("PD-05", "PD-05B", "Claim B: Net version net = expected_seed_denied, actionable_total_denied_amount NULL.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-06", "rows with payer_id -1 - scope " + SC, f"""
    SELECT k.payer_id, COALESCE(MAX(p.payer_name), 'Unknown Payer') AS payer,
           COUNT(DISTINCT k.claim_id) AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END) AS denied,
           COALESCE(SUM(k.{amt(SC)}), 0) AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {where()}
      AND k.payer_id = -1
    GROUP BY k.payer_id
    """, expect="0 or 1 row. When present, payer_id -1 is shown on the dashboard as Unknown Payer with the same values.")

In [ ]:
show("PD-07", "distinct payer_type values", f"""
SELECT k.payer_type, COUNT(*) AS versions
FROM {KPI} k
{where()}
GROUP BY k.payer_type
ORDER BY versions DESC
""", expect="Claim filing indicator classes or 'Unknown / Other'. No NULL or blank value.")

In [ ]:
_candidates = show("PD-08", "claims with several denied Gross versions", f"""
SELECT k.claim_id, COUNT(DISTINCT k.claim_version_id) AS gross_versions, SUM(k.gross_total_denied_amount) AS gross_denied
FROM {KPI} k
WHERE k.organization_id = {ORG} AND k.gross_total_denied_amount <> 0
GROUP BY k.claim_id
HAVING COUNT(DISTINCT k.claim_version_id) > 1
ORDER BY gross_versions DESC, k.claim_id
""", expect="Candidate claims. The first one is used for the trace cell unless CLAIM_IDS['PD-08'] is set.", limit=50)
pick("PD-08", _candidates)

In [ ]:
trace_claim("PD-08", "PD-08", "One KPI row per surviving Gross version with its own gross amount; exactly one version has is_net_version = Y and its net amount equals its gross amount; the claim counts once in Denied for its payer.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where(date_from=T12_FROM, date_to=TO_TODAY, payer_types=[], payer_ids=[], claim_types=[])
    show("PD-09", "default view - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Scorecard Claims, Denied, Rate and Charges per payer with default filters.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where(date_from=MTD_FROM, date_to=TO_TODAY)
    show("PD-10", "MTD - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Widgets with Date range = MTD.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where()
    show("PD-10", "custom range - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Widgets with the custom range FROM_DATE to TO_DATE.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where(payer_types=[FILTER_PAYER_TYPE])
    show("PD-11", "payer type " + FILTER_PAYER_TYPE + " - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Only payers of the selected type, with these values.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where(payer_ids=[FILTER_PAYER_ID])
    show("PD-11", "single payer - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Only the selected payer, with these values.")

In [ ]:
show("PD-12", "distinct claim_type values", f"""
SELECT k.claim_type, COUNT(*) AS versions
FROM {KPI} k
WHERE k.organization_id = {ORG}
GROUP BY k.claim_type
""", expect="Professional and Institutional; Unknown only where the R1 claim-type rule cannot be applied (Business Rules 11/20, Topic 2).")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where(claim_types=[FILTER_CLAIM_TYPE])
    show("PD-12", "claim type - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Widgets with the selected claim type.")

In [ ]:
SC = "NET"
W = where()
show("PD-13", "All denials - scope " + SC, f"""
SELECT k.payer_id,
       COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
       COUNT(DISTINCT k.claim_id)                                                     AS claims,
       COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
       ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                 / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
       COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
FROM {KPI} k
LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
{W}
GROUP BY k.payer_id
ORDER BY charges DESC
""",
     expect="Widgets with Denial scope = All denials.")

In [ ]:
SC = "ACTIONABLE"
W = where()
show("PD-13", "Actionable - scope " + SC, f"""
SELECT k.payer_id,
       COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
       COUNT(DISTINCT k.claim_id)                                                     AS claims,
       COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
       ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                 / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
       COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
FROM {KPI} k
LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
{W}
GROUP BY k.payer_id
ORDER BY charges DESC
""",
     expect="Widgets with Denial scope = Actionable denials only.")

In [ ]:
show("PD-13", "payers where Claims differ between scopes or Actionable exceeds Net", f"""
SELECT k.payer_id,
       COUNT(DISTINCT CASE WHEN {denied_cond('NET')} THEN k.claim_id END)        AS net_denied,
       COUNT(DISTINCT CASE WHEN {denied_cond('ACTIONABLE')} THEN k.claim_id END) AS actionable_denied,
       COALESCE(SUM(k.net_total_denied_amount), 0)                              AS net_charges,
       COALESCE(SUM(k.actionable_total_denied_amount), 0)                       AS actionable_charges
FROM {KPI} k
{where()}
GROUP BY k.payer_id
HAVING actionable_denied > net_denied OR actionable_charges > net_charges
""", expect="0 rows (Actionable never exceeds Net; Claims is the same query for both scopes).", expect_zero_rows=True)

In [ ]:
SC = "GROSS"
W = where()
show("PD-14", "Gross - scope " + SC, f"""
SELECT k.payer_id,
       COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
       COUNT(DISTINCT k.claim_id)                                                     AS claims,
       COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
       ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                 / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
       COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
FROM {KPI} k
LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
{W}
GROUP BY k.payer_id
ORDER BY charges DESC
""",
     expect="Widgets with Denial scope = Cumulative submissions, if the option is offered on the Payer page.")

In [ ]:
visual("PD-15", "switching Dollars / Claims / both changes which charts are shown, not their values (compare with PD-09)")

In [ ]:
visual("PD-16", "the facility filter is hidden or does not change any value")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-17", "denied charges by payer type - scope " + SC, f"""
    WITH t AS (
      SELECT k.payer_type,
             COALESCE(SUM(k.{amt(SC)}), 0)                                      AS denied_charges,
             COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)   AS denied_claims
      FROM {KPI} k
      {where()}
      GROUP BY k.payer_type)
    SELECT payer_type, denied_charges,
           ROUND(100 * denied_charges / NULLIF(SUM(denied_charges) OVER (), 0), 0) AS charges_pct
    FROM t
    ORDER BY 2 DESC
    """,
         expect="Each slice amount and percentage; the centre total is the sum of denied_charges.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-18", "denied claims by payer type - scope " + SC, f"""
    WITH t AS (
      SELECT k.payer_type,
             COALESCE(SUM(k.{amt(SC)}), 0)                                      AS denied_charges,
             COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)   AS denied_claims
      FROM {KPI} k
      {where()}
      GROUP BY k.payer_type)
    SELECT payer_type, denied_claims,
           ROUND(100 * denied_claims / NULLIF(SUM(denied_claims) OVER (), 0), 0) AS claims_pct
    FROM t
    ORDER BY 2 DESC
    """,
         expect="Each slice count and percentage.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where()
    show("PD-19", "denied charges by payer - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Each bar = charges for the payer.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where()
    show("PD-20", "denied claims by payer - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Each bar = denied for the payer.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-21", "claims, denied and accepted per payer - scope " + SC, f"""
    SELECT k.payer_id,
           COUNT(DISTINCT k.claim_id)                                                            AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                         AS denied,
           COUNT(DISTINCT CASE WHEN k.claim_rollup_type = 'Non-Denial' THEN k.claim_id END)      AS non_denial_claims
    FROM {KPI} k
    {where()}
    GROUP BY k.payer_id
    ORDER BY claims DESC
    """, expect="Scorecard Claims = claims. claims >= denied; non_denial_claims are included in claims.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where()
    show("PD-22", "Denied and Rate - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Denied and Rate (one decimal) per payer for the scope.")

In [ ]:
SC = "ACTIONABLE"
W = where()
show("PD-22", "Denied and Rate - scope " + SC, f"""
SELECT k.payer_id,
       COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
       COUNT(DISTINCT k.claim_id)                                                     AS claims,
       COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
       ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                 / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
       COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
FROM {KPI} k
LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
{W}
GROUP BY k.payer_id
ORDER BY charges DESC
""",
     expect="Denied and Rate (one decimal) per payer, Actionable. Claims equal the All denials result.")

In [ ]:
for SC in SCOPES_TO_TEST:
    W = where()
    show("PD-23", "Charges - scope " + SC, f"""
    SELECT k.payer_id,
           COALESCE(MAX(p.payer_name), 'Unknown Payer')                                   AS payer,
           COUNT(DISTINCT k.claim_id)                                                     AS claims,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)                AS denied,
           ROUND(100 * COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)
                     / NULLIF(COUNT(DISTINCT k.claim_id), 0), 1)                          AS rate_pct,
           COALESCE(SUM(k.{amt(SC)}), 0)                                                  AS charges
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {W}
    GROUP BY k.payer_id
    ORDER BY charges DESC
    """,
         expect="Charges per payer.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-24", "Top CARC per payer - scope " + SC, f"""
    WITH c AS (
      SELECT payer_id, adjustment_group_code, adjustment_reason_code, SUM(denied_amount) AS amt
      FROM {S_CARC}
      WHERE scope = '{SC}' AND organization_id = {ORG}
        AND production_date BETWEEN DATE'{FROM_DATE}' AND DATE'{TO_DATE}'
      GROUP BY payer_id, adjustment_group_code, adjustment_reason_code),
    r AS (SELECT c.*, ROW_NUMBER() OVER (PARTITION BY payer_id ORDER BY amt DESC) AS rn FROM c WHERE amt IS NOT NULL)
    SELECT payer_id, CONCAT('CARC ', adjustment_reason_code) AS top_carc, adjustment_group_code, amt
    FROM r WHERE rn = 1
    ORDER BY amt DESC
    """, expect="Top CARC per payer, displayed as 'CARC <reason>'.")

In [ ]:
for SC in SCOPES_TO_TEST:
    try:
        _pcarc_exists = spark.catalog.tableExists(PCARC)
    except Exception:
        _pcarc_exists = False
    _pcarc_rows = 0
    if _pcarc_exists:
        try:
            _pcarc_rows = spark.sql(f"SELECT COUNT(*) FROM {PCARC} WHERE organization_id = {ORG}").collect()[0][0]
        except Exception:
            _pcarc_rows = 0
    if not _pcarc_exists or _pcarc_rows == 0:
        skipped("PD-24", "Top CARC from R2_DENIAL_CLAIM_PAYER_CARC - scope " + SC,
                f"{PCARC} is " + ("empty for this organization" if _pcarc_exists else "not available") + " - dashboard check stays Blocked")
    else:
        show("PD-24", "Top CARC per payer from R2_DENIAL_CLAIM_PAYER_CARC - scope " + SC, f"""
        WITH c AS (
          SELECT payer_id, adjustment_group_code, adjustment_reason_code,
                 SUM({ {'NET': 'net_denied_amount', 'ACTIONABLE': 'actionable_denied_amount', 'GROSS': 'gross_denied_amount'}[SC] }) AS amt
          FROM {PCARC}
          WHERE organization_id = {ORG} AND adjustment_reason_code IS NOT NULL
            AND production_date BETWEEN DATE'{FROM_DATE}' AND DATE'{TO_DATE}'
          GROUP BY payer_id, adjustment_group_code, adjustment_reason_code),
        r AS (SELECT c.*, ROW_NUMBER() OVER (PARTITION BY payer_id ORDER BY amt DESC) AS rn FROM c WHERE amt IS NOT NULL)
        SELECT payer_id, CONCAT('CARC ', adjustment_reason_code) AS top_carc, adjustment_group_code, amt
        FROM r WHERE rn = 1
        ORDER BY amt DESC
        """, expect="Same result as the semantic-layer cell above.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-25", "month-over-month change per payer (one candidate definition) - scope " + SC, f"""
    WITH m AS (
      SELECT k.payer_id, TRUNC(k.production_date, 'MM') AS month, COALESCE(SUM(k.{amt(SC)}), 0) AS charges
      FROM {KPI} k
      {where()}
      GROUP BY k.payer_id, TRUNC(k.production_date, 'MM')),
    r AS (SELECT m.*, ROW_NUMBER() OVER (PARTITION BY payer_id ORDER BY month DESC) AS rn FROM m)
    SELECT cur.payer_id, prv.month AS previous_month, prv.charges AS previous_charges,
           cur.month AS last_month, cur.charges AS last_charges,
           ROUND(100 * (cur.charges - prv.charges) / NULLIF(prv.charges, 0), 1) AS change_pct
    FROM r cur LEFT JOIN r prv ON prv.payer_id = cur.payer_id AND prv.rn = 2
    WHERE cur.rn = 1
    ORDER BY cur.payer_id
    """, expect="Use once the Trend definition is agreed (Open Points). Shown here: last month vs previous month.")

In [ ]:
visual("PD-26", "the Historical recovery column is hidden or shows no calculated value")

In [ ]:
visual("PD-27", "sorting, paging and search do not change values; a row click opens the payer drill-down")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-28", "denied measure by payer and month - scope " + SC, f"""
    SELECT COALESCE(p.payer_name, 'Unknown Payer')                                 AS payer,
           TRUNC(k.production_date, 'MM')                                          AS month,
           COALESCE(SUM(k.{amt(SC)}), 0)                                             AS denied_charges,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)           AS denied_claims
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {where()}
    GROUP BY COALESCE(p.payer_name, 'Unknown Payer'), TRUNC(k.production_date, 'MM')
    ORDER BY payer, month
    """,
         expect="Each point of each payer line.", limit=1000)

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-29", "payer rank within each month (dollars basis) - scope " + SC, f"""
    WITH m AS (
      SELECT COALESCE(p.payer_name, 'Unknown Payer') AS payer, TRUNC(k.production_date, 'MM') AS month,
             COALESCE(SUM(k.{amt(SC)}), 0) AS denied_charges,
             COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END) AS denied_claims
      FROM {KPI} k
      LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
      {where()}
      GROUP BY COALESCE(p.payer_name, 'Unknown Payer'), TRUNC(k.production_date, 'MM'))
    SELECT month, payer, denied_charges,
           RANK() OVER (PARTITION BY month ORDER BY denied_charges DESC) AS rank_by_charges,
           denied_claims,
           RANK() OVER (PARTITION BY month ORDER BY denied_claims DESC)  AS rank_by_claims
    FROM m
    ORDER BY month, rank_by_charges
    """, expect="Each payer's rank in each month (rank 1 = highest).", limit=1000)

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-30", "heatmap cells - scope " + SC, f"""
    SELECT COALESCE(p.payer_name, 'Unknown Payer')                                 AS payer,
           TRUNC(k.production_date, 'MM')                                          AS month,
           COALESCE(SUM(k.{amt(SC)}), 0)                                             AS denied_charges,
           COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END)           AS denied_claims
    FROM {KPI} k
    LEFT JOIN {DPAY} p ON p.payer_id = k.payer_id
    {where()}
    GROUP BY COALESCE(p.payer_name, 'Unknown Payer'), TRUNC(k.production_date, 'MM')
    ORDER BY payer, month
    """,
         expect="Each cell of the heatmap.", limit=1000)

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-30", "payers whose monthly cells do not add up to their total - scope " + SC, f"""
    WITH cells AS (
      SELECT k.payer_id, TRUNC(k.production_date, 'MM') AS month, COALESCE(SUM(k.{amt(SC)}), 0) AS charges
      FROM {KPI} k
      {where()}
      GROUP BY k.payer_id, TRUNC(k.production_date, 'MM')),
    rowsum AS (SELECT payer_id, SUM(charges) AS heatmap_total FROM cells GROUP BY payer_id),
    total AS (
      SELECT k.payer_id, COALESCE(SUM(k.{amt(SC)}), 0) AS scorecard_charges
      FROM {KPI} k
      {where()}
      GROUP BY k.payer_id)
    SELECT t.payer_id, t.scorecard_charges, r.heatmap_total
    FROM total t JOIN rowsum r ON r.payer_id = t.payer_id
    WHERE t.scorecard_charges <> r.heatmap_total
    """, expect="0 rows.", expect_zero_rows=True)

In [ ]:
for SC in SCOPES_TO_TEST:
    CAT_AMT = amt(SC)
    show("PD-31", "categories for payer " + str(DRILL_PAYER_ID) + " - scope " + SC, f"""
    WITH c AS (
      SELECT c.denial_category,
             COALESCE(SUM(c.{CAT_AMT}), 0)                                                AS denied_charges,
             COUNT(DISTINCT CASE WHEN c.{CAT_AMT} IS NOT NULL THEN c.claim_id END)       AS denied_claims
      FROM {CAT} c
      {where(alias='c', payer_ids=[DRILL_PAYER_ID])}
      GROUP BY c.denial_category)
    SELECT denial_category, denied_charges,
           ROUND(100 * denied_charges / NULLIF(SUM(denied_charges) OVER (), 0), 1) AS share_pct,
           denied_claims
    FROM c
    ORDER BY denied_charges DESC
    """, expect="Categories, amounts and shares of the drill-down 'By category' view.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-31", "category total vs KPI total for payer " + str(DRILL_PAYER_ID) + " - scope " + SC, f"""
    SELECT (SELECT COALESCE(SUM(c.{amt(SC)}), 0) FROM {CAT} c {where(alias='c', payer_ids=[DRILL_PAYER_ID])}) AS category_total,
           (SELECT COALESCE(SUM(k.{amt(SC)}), 0) FROM {KPI} k {where(payer_ids=[DRILL_PAYER_ID])})          AS scorecard_charges
    """, expect="category_total = scorecard_charges.")

In [ ]:
visual("PD-32", "By CARC / RARC, bulletins, ledger and plans are hidden or marked not available")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-33", "page totals computed three ways - scope " + SC, f"""
    WITH base AS (SELECT k.* FROM {KPI} k {where()}),
    by_type  AS (SELECT payer_type, SUM({amt(SC)}) AS a FROM base GROUP BY payer_type),
    by_payer AS (SELECT payer_id, SUM({amt(SC)}) AS a FROM base GROUP BY payer_id),
    by_month AS (SELECT payer_id, TRUNC(production_date, 'MM') AS m, SUM({amt(SC)}) AS a FROM base GROUP BY payer_id, TRUNC(production_date, 'MM'))
    SELECT (SELECT COALESCE(SUM(a), 0) FROM by_type)  AS donut_total,
           (SELECT COALESCE(SUM(a), 0) FROM by_payer) AS bars_and_scorecard_total,
           (SELECT COALESCE(SUM(a), 0) FROM by_month) AS heatmap_total
    """, expect="The three totals are equal and match the dashboard.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-34", "KPI dashboard totals vs sum over payers - scope " + SC, f"""
    WITH base AS (SELECT k.* FROM {KPI} k {where()}),
    per_payer AS (
      SELECT payer_id, COUNT(DISTINCT CASE WHEN {denied_cond(SC, alias='base')} THEN claim_id END) AS denied, SUM({amt(SC)}) AS charges
      FROM base GROUP BY payer_id)
    SELECT (SELECT COALESCE(SUM(base.{amt(SC)}), 0) FROM base)                                                AS kpi_charges_denied,
           (SELECT COALESCE(SUM(charges), 0) FROM per_payer)                                                AS payer_charges_total,
           (SELECT COUNT(DISTINCT CASE WHEN {denied_cond(SC, alias='base')} THEN claim_id END) FROM base)        AS kpi_claims_denied,
           (SELECT COALESCE(SUM(denied), 0) FROM per_payer)                                                 AS payer_denied_total
    """, expect="Charges equal. Denied equal unless a claim has versions under several payers (next cell).")

In [ ]:
show("PD-34", "claims spanning payers", f"""
SELECT k.claim_id, COUNT(DISTINCT k.payer_id) AS payers, CONCAT_WS(', ', COLLECT_SET(CAST(k.payer_id AS STRING))) AS payer_ids
FROM {KPI} k
{where()}
GROUP BY k.claim_id
HAVING COUNT(DISTINCT k.payer_id) > 1
""", expect="Lists the claims that explain any difference in the Denied totals of the previous cell.")

In [ ]:
show("PD-35", "payers where category totals differ from KPI totals", f"""
WITH k AS (
  SELECT payer_id, COALESCE(SUM(net_total_denied_amount), 0) AS net_amt,
         COALESCE(SUM(actionable_total_denied_amount), 0) AS act_amt, COALESCE(SUM(gross_total_denied_amount), 0) AS gross_amt
  FROM {KPI} WHERE organization_id = {ORG} GROUP BY payer_id),
c AS (
  SELECT payer_id, COALESCE(SUM(net_total_denied_amount), 0) AS net_amt,
         COALESCE(SUM(actionable_total_denied_amount), 0) AS act_amt, COALESCE(SUM(gross_total_denied_amount), 0) AS gross_amt
  FROM {CAT} WHERE organization_id = {ORG} GROUP BY payer_id)
SELECT COALESCE(k.payer_id, c.payer_id) AS payer_id,
       k.net_amt, c.net_amt AS cat_net, k.act_amt, c.act_amt AS cat_act, k.gross_amt, c.gross_amt AS cat_gross
FROM k FULL OUTER JOIN c ON c.payer_id = k.payer_id
WHERE COALESCE(k.net_amt, 0) <> COALESCE(c.net_amt, 0)
   OR COALESCE(k.act_amt, 0) <> COALESCE(c.act_amt, 0)
   OR COALESCE(k.gross_amt, 0) <> COALESCE(c.gross_amt, 0)
""", expect="0 rows.", expect_zero_rows=True)

In [ ]:
show("PD-36", "target and staging row counts for the organization", f"""
SELECT 'R2_KPI_DASHBOARD_DETAILS' AS tbl,
       (SELECT COUNT(*) FROM {ADB_KPI} WHERE organization_id = {ORG}) AS target_rows,
       (SELECT COUNT(*) FROM {KPI_STG} WHERE organization_id = {ORG}) AS staging_rows
UNION ALL
SELECT 'R2_DENIAL_CLAIM_CATEGORY',
       (SELECT COUNT(*) FROM {ADB_CAT} WHERE organization_id = {ORG}),
       (SELECT COUNT(*) FROM {CAT_STG} WHERE organization_id = {ORG})
""", expect="staging_rows = 0 for both tables. target_rows are compared with the next cell.")

In [ ]:
show("PD-36", "rows the ADB targets should hold", f"""
SELECT (SELECT COUNT(*) FROM (SELECT DISTINCT organization_id, claim_version_id FROM {S_KPI} WHERE organization_id = {ORG}) v)  AS kpi_target_rows_expected,
       (SELECT COUNT(*) FROM (SELECT DISTINCT organization_id, claim_version_id, denial_category FROM {S_CARC} WHERE organization_id = {ORG}) v)
                                                                                                                               AS category_target_rows_expected,
       (SELECT MAX(etl_run_id) FROM {S_KPI} WHERE organization_id = {ORG})                                                       AS kpi_run_id,
       (SELECT MAX(etl_run_id) FROM {S_CARC} WHERE organization_id = {ORG})                                                      AS carc_run_id
""", expect="Expected counts equal target_rows of the previous cell; both run ids are the same.")

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-37", "payers where ADB and the semantic layer differ - scope " + SC, f"""
    WITH s AS (
      SELECT payer_id,
             COUNT(DISTINCT CASE WHEN claim_rollup_type = 'Denial' THEN claim_id END) AS denied,
             COALESCE(SUM(total_denied_amount), 0) AS charges
      FROM {S_KPI}
      WHERE organization_id = {ORG} AND scope = '{SC}'
        AND production_date BETWEEN DATE'{FROM_DATE}' AND DATE'{TO_DATE}'
      GROUP BY payer_id),
    a AS (
      SELECT k.payer_id,
             COUNT(DISTINCT CASE WHEN {denied_cond(SC)} THEN k.claim_id END) AS denied,
             COALESCE(SUM(k.{amt(SC)}), 0) AS charges
      FROM {ADB_KPI} k
      WHERE k.organization_id = {ORG}
        AND k.production_date BETWEEN DATE'{FROM_DATE}' AND DATE'{TO_DATE}'
      GROUP BY k.payer_id)
    SELECT COALESCE(s.payer_id, a.payer_id) AS payer_id, s.denied AS semantic_denied, a.denied AS adb_denied,
           s.charges AS semantic_charges, a.charges AS adb_charges
    FROM s FULL OUTER JOIN a ON a.payer_id = s.payer_id
    WHERE COALESCE(s.denied, 0) <> COALESCE(a.denied, 0) OR COALESCE(s.charges, 0) <> COALESCE(a.charges, 0)
    """, expect="0 rows.", expect_zero_rows=True)

In [ ]:
show("PD-38", "claim versions present in Net and Gross whose denied amounts differ", f"""
WITH n AS (SELECT claim_version_id, claim_id, payer_id, total_denied_amount AS net_amt FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'NET'),
     g AS (SELECT claim_version_id, total_denied_amount AS gross_amt FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS')
SELECT COUNT(*)                                                                   AS versions_in_both_scopes,
       SUM(CASE WHEN n.net_amt <> g.gross_amt THEN 1 ELSE 0 END)                 AS versions_with_different_amounts,
       SUM(CASE WHEN n.net_amt > g.gross_amt THEN n.net_amt - g.gross_amt END)    AS net_above_gross_total,
       SUM(CASE WHEN n.net_amt < g.gross_amt THEN g.gross_amt - n.net_amt END)    AS gross_above_net_total
FROM n JOIN g ON g.claim_version_id = n.claim_version_id
""", expect="versions_with_different_amounts = 0: the same physical version should carry the same selected CAS in both scopes.")

In [ ]:
_diff = show("PD-38", "claim versions with the largest Net vs Gross difference", f"""
WITH n AS (SELECT claim_version_id, claim_id, payer_id, total_denied_amount AS net_amt FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'NET'),
     g AS (SELECT claim_version_id, total_denied_amount AS gross_amt FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS'),
     v AS (SELECT claim_id, COUNT(DISTINCT claim_version_id) AS gross_versions FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS' GROUP BY claim_id)
SELECT n.claim_id, n.claim_version_id, n.payer_id, n.net_amt, g.gross_amt, n.net_amt - g.gross_amt AS difference, v.gross_versions
FROM n JOIN g ON g.claim_version_id = n.claim_version_id
LEFT JOIN v ON v.claim_id = n.claim_id
WHERE n.net_amt <> g.gross_amt
ORDER BY ABS(n.net_amt - g.gross_amt) DESC
LIMIT 50
""", expect="0 rows. If rows appear, the last PD-38 cells show the Net and Gross rows of the claim with the largest difference.")
if _diff is not None and _diff.limit(1).count() and not CLAIM_IDS.get("PD-38"):
    CLAIM_IDS["PD-38"] = _diff.select("claim_id").first()[0]

In [ ]:
show("PD-38", "claim versions whose denial status differs between Net and Gross", f"""
WITH n AS (SELECT claim_version_id, claim_rollup_type AS net_rollup, denial_type AS net_denial_type
           FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'NET'),
     g AS (SELECT claim_version_id, claim_rollup_type AS gross_rollup, denial_type AS gross_denial_type
           FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS')
SELECT n.net_rollup, g.gross_rollup, n.net_denial_type, g.gross_denial_type, COUNT(*) AS versions
FROM n JOIN g ON g.claim_version_id = n.claim_version_id
WHERE NOT (n.net_rollup <=> g.gross_rollup) OR NOT (n.net_denial_type <=> g.gross_denial_type)
GROUP BY n.net_rollup, g.gross_rollup, n.net_denial_type, g.gross_denial_type
ORDER BY versions DESC
""", expect="0 rows: the same claim version has the same denial status in Net and Gross.", expect_zero_rows=True)

In [ ]:
show("PD-38", "Net vs Gross attribute differences on the same claim version", f"""
WITH n AS (SELECT * FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'NET'),
     g AS (SELECT * FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS'),
d AS (
SELECT SUM(CASE WHEN NOT (n.claim_id <=> g.claim_id) THEN 1 ELSE 0 END)                       AS claim_id_diff,
       SUM(CASE WHEN NOT (n.production_date <=> g.production_date) THEN 1 ELSE 0 END)         AS production_date_diff,
       SUM(CASE WHEN NOT (n.payer_id <=> g.payer_id) THEN 1 ELSE 0 END)                       AS payer_id_diff,
       SUM(CASE WHEN NOT (n.payer_type <=> g.payer_type) THEN 1 ELSE 0 END)                   AS payer_type_diff,
       SUM(CASE WHEN NOT (n.claim_type <=> g.claim_type) THEN 1 ELSE 0 END)                   AS claim_type_diff,
       SUM(CASE WHEN NOT (n.claim_charge_amount <=> g.claim_charge_amount) THEN 1 ELSE 0 END) AS charge_diff,
       SUM(CASE WHEN NOT (n.patient_control_number <=> g.patient_control_number) THEN 1 ELSE 0 END) AS pcn_diff,
       COUNT(*) AS versions_compared
FROM n JOIN g ON g.claim_version_id = n.claim_version_id)
SELECT * FROM d WHERE claim_id_diff + production_date_diff + payer_id_diff + payer_type_diff + claim_type_diff + charge_diff + pcn_diff > 0
""", expect="0 rows: no attribute differs between the Net row and the Gross row of the same version.", expect_zero_rows=True)

In [ ]:
show("PD-38", "Net claim versions that are not surviving Gross versions", f"""
SELECT n.claim_id, n.claim_version_id, n.production_date, n.total_denied_amount
FROM {S_KPI} n
WHERE n.organization_id = {ORG} AND n.scope = 'NET'
  AND n.claim_version_id NOT IN (SELECT claim_version_id FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS')
ORDER BY n.claim_id
""", expect="0 rows: every Net version is a surviving Gross version (no Net-only versions).", expect_zero_rows=True, limit=50)

In [ ]:
CID = claim("PD-38")
if CID is None:
    skipped("PD-38", "Net and Gross versions of the worst claim", "no Net vs Gross difference found")
else:
    show("PD-38", "Net and Gross KPI rows of claim " + CID, f"""
    SELECT claim_version_id, scope, production_date, payer_id, claim_rollup_type, denial_type,
           claim_charge_amount, total_denied_amount, actionable_denied_amount, etl_run_id
    FROM {S_KPI}
    WHERE organization_id = {ORG} AND claim_id = '{CID}'
    ORDER BY production_date, claim_version_id, scope
    """, expect="Each version once per scope it belongs to. Compare total_denied_amount of the same version across scopes.")

In [ ]:
CID = claim("PD-38")
if CID is None:
    skipped("PD-38", "Net and Gross CARC rows of the worst claim", "no Net vs Gross difference found")
else:
    show("PD-38", "Net and Gross CARC rows of claim " + CID, f"""
    SELECT claim_version_id, scope, adjustment_group_code, adjustment_reason_code, denial_category,
           COUNT(*) AS cas_rows, SUM(denied_amount) AS denied_amount, SUM(actionable_denied_amount) AS actionable_denied_amount
    FROM {S_CARC}
    WHERE organization_id = {ORG} AND claim_id = '{CID}'
    GROUP BY claim_version_id, scope, adjustment_group_code, adjustment_reason_code, denial_category
    ORDER BY claim_version_id, adjustment_group_code, adjustment_reason_code, scope
    """, expect="The CAS that make up the Net and the Gross amount of each version. A CARC present in one scope only explains the difference.")

In [ ]:
show("PD-39", "Net and Gross claim populations", f"""
WITH n AS (SELECT claim_id, COUNT(DISTINCT claim_version_id) AS net_versions FROM {S_KPI}
           WHERE organization_id = {ORG} AND scope = 'NET' GROUP BY claim_id),
     g AS (SELECT DISTINCT claim_id FROM {S_KPI} WHERE organization_id = {ORG} AND scope = 'GROSS')
SELECT (SELECT COUNT(*) FROM n)                                                   AS net_claims,
       (SELECT COUNT(*) FROM g)                                                   AS gross_claims,
       (SELECT COUNT(*) FROM n WHERE net_versions > 1)                            AS claims_with_several_net_versions,
       (SELECT COUNT(*) FROM g WHERE claim_id NOT IN (SELECT claim_id FROM n))    AS gross_claims_without_net,
       (SELECT COUNT(*) FROM n WHERE claim_id NOT IN (SELECT claim_id FROM g))    AS net_claims_without_gross
""", expect="net_claims = gross_claims; claims_with_several_net_versions = 0; gross_claims_without_net = 0; net_claims_without_gross = 0.")

In [ ]:
show("PD-39", "claims with several Net versions, or present in one scope only", f"""
WITH n AS (SELECT claim_id, COUNT(DISTINCT claim_version_id) AS net_versions FROM {S_KPI}
           WHERE organization_id = {ORG} AND scope = 'NET' GROUP BY claim_id),
     g AS (SELECT claim_id, COUNT(DISTINCT claim_version_id) AS gross_versions FROM {S_KPI}
           WHERE organization_id = {ORG} AND scope = 'GROSS' GROUP BY claim_id)
SELECT COALESCE(n.claim_id, g.claim_id) AS claim_id, n.net_versions, g.gross_versions
FROM n FULL OUTER JOIN g ON g.claim_id = n.claim_id
WHERE n.claim_id IS NULL OR g.claim_id IS NULL OR n.net_versions > 1
ORDER BY 1
""", expect="0 rows.", expect_zero_rows=True, limit=50)

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW g_net_ranked AS
WITH c AS (
  SELECT v.*
  FROM g_versions v
  WHERE v.claim_version_id IN (SELECT claim_version_id FROM g_expected
                               WHERE COALESCE(payment_amount, 0) >= 0 AND COALESCE(claim_charge, 0) > 0
                                 AND COALESCE(line_rule_breach, 0) = 0
                                 AND claim_version_id NOT IN (SELECT claim_version_id FROM g_cas WHERE amt < 0)
                                 AND NOT (selected_level = 'CLAIM' AND seed_denied_amount > claim_charge))),
r AS (
  SELECT c.*,
    CASE WHEN claim_status_code = '4' THEN 1 WHEN claim_status_code = '3' THEN 2 WHEN claim_status_code = '2' THEN 3
         WHEN claim_status_code IN ('19','20','21') THEN 4 WHEN claim_status_code = '1' THEN 5 ELSE 6 END AS status_rank_email,
    CASE WHEN claim_status_code IN ('1','2','3','19','20','21','23') THEN 1 WHEN claim_status_code = '4' THEN 2
         WHEN claim_status_code = '22' THEN 3 ELSE 4 END AS status_rank_mapping
  FROM c)
SELECT r.*,
  ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY production_dt DESC NULLS LAST, REGEXP_REPLACE(pccn_norm, '\\\\s+', ' ') ASC NULLS LAST,
                     status_rank_email ASC, claim_version_id ASC) AS rn_email,
  ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY version_ts DESC NULLS LAST, rec_insert_ts DESC NULLS LAST,
                     production_dt DESC NULLS LAST, status_rank_mapping ASC, header_id ASC NULLS LAST, claim_version_id ASC) AS rn_mapping,
  COUNT(*) OVER (PARTITION BY claim_id) AS eligible_versions
FROM r
""")
show("PD-40", "selected Net version vs the expected winner (both precedence rules)", f"""
SELECT COUNT(*)                                                                        AS net_claims_checked,
       SUM(CASE WHEN eligible_versions > 1 THEN 1 ELSE 0 END)                          AS claims_with_several_versions,
       SUM(CASE WHEN w_email.claim_version_id <> n.claim_version_id THEN 1 ELSE 0 END)   AS differs_from_email_rule,
       SUM(CASE WHEN w_map.claim_version_id <> n.claim_version_id THEN 1 ELSE 0 END)     AS differs_from_mapping_rule,
       SUM(CASE WHEN w_email.claim_version_id IS NULL THEN 1 ELSE 0 END)                AS net_claims_without_eligible_gold_version
FROM n_sel n
LEFT JOIN (SELECT claim_id, claim_version_id, eligible_versions FROM g_net_ranked WHERE rn_email = 1) w_email ON w_email.claim_id = n.claim_id
LEFT JOIN (SELECT claim_id, claim_version_id FROM g_net_ranked WHERE rn_mapping = 1) w_map ON w_map.claim_id = n.claim_id
""", expect=f"differs_from_{EXPECTED_PRECEDENCE.lower()}_rule = 0 (EXPECTED_PRECEDENCE = {EXPECTED_PRECEDENCE}). The other rule is reported for the open point on precedence. Eligible = positive payment, charge > 0, no denied-over-charge breach, no negative CAS at either level (as the load notebook's NEGATIVE_CAS_AMOUNT rule).")

In [ ]:
_rn = "rn_email" if EXPECTED_PRECEDENCE == "EMAIL" else "rn_mapping"
_p40 = show("PD-40", "claims whose Net version is not the expected winner (" + EXPECTED_PRECEDENCE + ")", f"""
SELECT n.claim_id, n.claim_version_id AS net_version, w.claim_version_id AS expected_version,
       nv.production_dt AS net_production_date, w.production_dt AS expected_production_date,
       nv.claim_status_code AS net_status, w.claim_status_code AS expected_status,
       nv.pccn_norm AS net_pccn, w.pccn_norm AS expected_pccn, w.eligible_versions
FROM n_sel n
JOIN (SELECT * FROM g_net_ranked WHERE {_rn} = 1) w ON w.claim_id = n.claim_id
LEFT JOIN g_versions nv ON nv.claim_version_id = n.claim_version_id
WHERE w.claim_version_id <> n.claim_version_id
ORDER BY n.claim_id
""", expect="0 rows.", expect_zero_rows=True, limit=50)
if _p40 is not None and _p40.limit(1).count() and not CLAIM_IDS.get("PD-40"):
    CLAIM_IDS["PD-40"] = _p40.select("claim_id").first()[0]

In [ ]:
CID = claim("PD-40")
if CID is None:
    skipped("PD-40", "Gold Gross versions of the first differing claim", "no differing claim")
else:
    show("PD-40", "Gold Gross versions of claim " + CID + " with both rankings", f"""
    SELECT v.claim_version_id, CASE WHEN n.claim_version_id IS NOT NULL THEN 'Y' ELSE 'N' END AS is_net_version,
           v.production_dt, v.pccn_norm, v.claim_status_code, v.version_ts, v.rec_insert_ts, v.header_id,
           r.rn_email, r.rn_mapping, e.payment_amount, e.claim_charge, e.seed_denied_amount
    FROM g_versions v
    LEFT JOIN g_net_ranked r ON r.claim_version_id = v.claim_version_id
    LEFT JOIN g_expected e ON e.claim_version_id = v.claim_version_id
    LEFT JOIN n_sel n ON n.claim_version_id = v.claim_version_id
    WHERE v.claim_id = '{CID}'
    ORDER BY r.rn_email NULLS LAST, v.claim_version_id
    """, expect="rn_email / rn_mapping = 1 marks each rule's winner; NULL rank = version not eligible (excluded).")

In [ ]:
show("PD-41", "dashboard versions that meet an exclusion rule", f"""
WITH k AS (SELECT DISTINCT scope, claim_version_id FROM {S_KPI} WHERE organization_id = {ORG}),
x AS (
  SELECT k.scope, e.*,
         CASE WHEN e.payment_amount < 0 AND COALESCE(e.claim_status_code, '') <> '22' THEN 1 ELSE 0 END AS r_negative_payment,
         CASE WHEN e.claim_charge IS NULL OR e.claim_charge = 0 THEN 1 ELSE 0 END AS r_charge_null_or_zero,
         CASE WHEN e.selected_level = 'CLAIM' AND e.seed_denied_amount > e.claim_charge THEN 1 ELSE 0 END AS r_claim_denied_over_charge,
         CASE WHEN COALESCE(e.line_rule_breach, 0) = 1 THEN 1 ELSE 0 END AS r_line_denied_over_line_charge,
         CASE WHEN e.has_negative_denial_cas = 1 OR EXISTS (SELECT 1 FROM g_cas n WHERE n.claim_version_id = e.claim_version_id AND n.amt < 0) THEN 1 ELSE 0 END AS r_negative_denial_cas
  FROM k JOIN g_expected e ON e.claim_version_id = k.claim_version_id)
SELECT scope,
       SUM(r_negative_payment)            AS negative_payment_not_22,
       SUM(r_charge_null_or_zero)         AS charge_null_or_zero,
       SUM(r_claim_denied_over_charge)    AS claim_level_denied_over_charge,
       SUM(r_line_denied_over_line_charge) AS line_denied_over_line_charge_or_no_line_charge,
       SUM(r_negative_denial_cas)         AS negative_cas_any_level
FROM x
GROUP BY scope
HAVING SUM(r_negative_payment + r_charge_null_or_zero + r_claim_denied_over_charge + r_line_denied_over_line_charge + r_negative_denial_cas) > 0
ORDER BY scope
""", expect="0 rows: no version that meets an exclusion rule is in the dashboard population (Net or Gross).", expect_zero_rows=True)

In [ ]:
show("PD-41", "sample dashboard versions that meet an exclusion rule", f"""
WITH k AS (SELECT DISTINCT scope, claim_version_id FROM {S_KPI} WHERE organization_id = {ORG}),
x AS (
  SELECT k.scope, e.*,
         CASE WHEN e.payment_amount < 0 AND COALESCE(e.claim_status_code, '') <> '22' THEN 1 ELSE 0 END AS r_negative_payment,
         CASE WHEN e.claim_charge IS NULL OR e.claim_charge = 0 THEN 1 ELSE 0 END AS r_charge_null_or_zero,
         CASE WHEN e.selected_level = 'CLAIM' AND e.seed_denied_amount > e.claim_charge THEN 1 ELSE 0 END AS r_claim_denied_over_charge,
         CASE WHEN COALESCE(e.line_rule_breach, 0) = 1 THEN 1 ELSE 0 END AS r_line_denied_over_line_charge,
         CASE WHEN e.has_negative_denial_cas = 1 OR EXISTS (SELECT 1 FROM g_cas n WHERE n.claim_version_id = e.claim_version_id AND n.amt < 0) THEN 1 ELSE 0 END AS r_negative_denial_cas
  FROM k JOIN g_expected e ON e.claim_version_id = k.claim_version_id)
SELECT scope, claim_id, claim_version_id, claim_status_code, payment_amount, claim_charge, selected_level,
       seed_denied_amount, seed_denial_cas, r_negative_payment, r_charge_null_or_zero, r_claim_denied_over_charge,
       r_line_denied_over_line_charge, r_negative_denial_cas
FROM x
WHERE r_negative_payment + r_charge_null_or_zero + r_claim_denied_over_charge + r_line_denied_over_line_charge + r_negative_denial_cas > 0
ORDER BY scope, claim_id
""", expect="0 rows.", expect_zero_rows=True, limit=50)

In [ ]:
EXCL_TABLE = f"{SEMANTIC_SCHEMA}.denial_claims_exclusions"
_xcols = _columns(EXCL_TABLE)
if not _xcols:
    skipped("PD-41", "denial_claims_exclusions summary", f"{EXCL_TABLE} is not available")
else:
    _reason = next((c for c in ("exclusion_reason", "review_reason", "reason", "exclusion_rule", "rule_id") if c in _xcols), None)
    _org = f"WHERE organization_id = {ORG}" if "organization_id" in _xcols else ""
    _grp = _reason or "'all rows'"
    show("PD-41", "excluded versions logged in denial_claims_exclusions", f"""
    SELECT {_grp} AS exclusion_reason, COUNT(*) AS rows_logged
    FROM {EXCL_TABLE} {_org}
    GROUP BY {_grp}
    ORDER BY rows_logged DESC
    """, expect="One group per exclusion rule; compare the counts with Gold Gross (previous cells) and confirm the excluded versions are absent from the KPI rows.")

In [ ]:
show("PD-42", "intermediate KPI rows whose denial type does not follow the amount / charge rule", f"""
SELECT scope, problem, COUNT(*) AS versions
FROM (
SELECT scope,
       CASE WHEN total_denied_amount IS NULL THEN 'amount NULL'
            WHEN total_denied_amount < 0 THEN 'negative amount'
            WHEN claim_charge_amount IS NULL OR claim_charge_amount = 0 THEN 'charge NULL or 0'
            WHEN total_denied_amount > claim_charge_amount THEN 'amount above charge'
            WHEN total_denied_amount = 0 AND denial_type <> 'Accepted' THEN 'amount 0 but not Accepted'
            WHEN total_denied_amount = claim_charge_amount AND denial_type <> 'Fully Denied' THEN 'amount = charge but not Fully Denied'
            WHEN total_denied_amount > 0 AND total_denied_amount < claim_charge_amount AND denial_type <> 'Partially Denied' THEN '0 < amount < charge but not Partially Denied'
            WHEN (denial_type IN ('Fully Denied', 'Partially Denied')) <> (claim_rollup_type = 'Denial') THEN 'rollup type inconsistent'
       END AS problem
FROM {S_KPI}
WHERE organization_id = {ORG}) t
WHERE problem IS NOT NULL
GROUP BY scope, problem
ORDER BY scope, versions DESC
""", expect="0 rows: Accepted = 0; Fully Denied = charge; Partially Denied between 0 and charge; Denial rollup only for Fully / Partially Denied; no negative amount or amount above charge.", expect_zero_rows=True)

In [ ]:
for SC in SCOPES_TO_TEST:
    show("PD-43", "intermediate denied amount vs Gold Gross seed recomputation - scope " + SC, f"""
    SELECT COUNT(*)                                                                                AS versions,
           SUM(CASE WHEN k.total_denied_amount = e.seed_denied_amount THEN 1 ELSE 0 END)         AS equal,
           SUM(CASE WHEN k.total_denied_amount <> e.seed_denied_amount THEN 1 ELSE 0 END)        AS different,
           SUM(CASE WHEN e.claim_version_id IS NULL THEN 1 ELSE 0 END)                           AS not_found_in_gold,
           SUM(CASE WHEN k.total_denied_amount = 0 AND e.seed_denied_amount > 0 THEN 1 ELSE 0 END) AS accepted_but_seed_denial,
           SUM(CASE WHEN k.total_denied_amount > 0 AND e.seed_denied_amount = 0 THEN 1 ELSE 0 END) AS denied_without_seed_denial,
           SUM(k.total_denied_amount)                                                            AS intermediate_denied,
           SUM(e.seed_denied_amount)                                                             AS gold_seed_denied
    FROM {S_KPI} k
    LEFT JOIN g_expected e ON e.claim_version_id = k.claim_version_id
    WHERE k.organization_id = {ORG} AND k.scope = '{SC}'
    """, expect="Most versions equal; differences limited to CARC-RARC refinements and review exclusions. Large accepted_but_seed_denial counts point to lost denials.")

In [ ]:
_p43 = show("PD-43", "claim versions with the largest difference (Net)", f"""
SELECT k.claim_id, k.claim_version_id, k.denial_type, k.total_denied_amount AS intermediate_amount,
       e.seed_denied_amount AS gold_seed_amount, k.total_denied_amount - e.seed_denied_amount AS difference,
       e.selected_level, e.seed_denial_cas, e.claim_charge
FROM {S_KPI} k
JOIN g_expected e ON e.claim_version_id = k.claim_version_id
WHERE k.organization_id = {ORG} AND k.scope = 'NET' AND k.total_denied_amount <> e.seed_denied_amount
ORDER BY ABS(k.total_denied_amount - e.seed_denied_amount) DESC, k.claim_id
LIMIT 50
""", expect="Each row explained by a CARC-RARC refinement or a review exclusion (next cells); otherwise a defect.")
if _p43 is not None and _p43.limit(1).count() and not CLAIM_IDS.get("PD-43"):
    CLAIM_IDS["PD-43"] = _p43.select("claim_id").first()[0]

In [ ]:
CID = claim("PD-43")
if CID is None:
    skipped("PD-43", "CAS and review rows of the first differing claim", "no difference found")
else:
    show("PD-43", "selected Gold Gross CAS of claim " + CID + " (Net version)", f"""
    SELECT c.claim_version_id, c.cas_level, c.line_id, c.line_charge, c.carc_key, c.amt, c.in_seed, c.seed_denial
    FROM g_cas c
    JOIN g_expected e ON e.claim_version_id = c.claim_version_id AND e.selected_level = c.cas_level
    JOIN n_sel n ON n.claim_version_id = c.claim_version_id
    WHERE n.claim_id = '{CID}'
    ORDER BY c.cas_level, c.line_id, c.carc_key
    """, expect="Compare with the CARC rows of the intermediate table and the review table for the claim.")
    show("PD-43", "intermediate CARC rows of claim " + CID, f"""
    SELECT scope, claim_version_id, adjustment_group_code, adjustment_reason_code, denial_category, denied_amount, actionable_denied_amount
    FROM {S_CARC} WHERE organization_id = {ORG} AND claim_id = '{CID}'
    ORDER BY scope, claim_version_id, adjustment_group_code, adjustment_reason_code
    """, expect="The CARC rows that make up the intermediate amount.")

In [ ]:
summary_df = spark.createDataFrame(
    [(i + 1, *r) for i, r in enumerate(RESULTS)],
    "seq INT, test_id STRING, check_name STRING, status STRING, rows_returned BIGINT, detail STRING",
)
for row in summary_df.groupBy("status").count().orderBy("status").collect():
    print(f"{row['status']:8} {row['count']}")
print("Claims used:", {k: v for k, v in CLAIM_IDS.items() if v})
try:
    display(summary_df.orderBy("seq"))
except NameError:
    summary_df.orderBy("seq").show(200, truncate=False)